# Milestone 3: First baseline

This notebook uses only the `1year` data, and thus we will be using the file named `1year`. We will use the training set to fit the models and the validation to determine the relative strength of each model. We are avoiding using the test set and will use it later for final evaluation.


In [34]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 1. Load the cleaned dataset

Each file has 64 financial ratios (inputs) and a `bankrupt` (output) column.
 1 and 0 represent bankrupt and non-bankrupt respectively.

In [35]:
train = pd.read_csv('data/cleaned/1year/1year_train.csv')
validation = pd.read_csv('data/cleaned/1year/1year_validation.csv')

X_train = train.drop(columns='bankrupt')
y_train = train['bankrupt']
X_validation = validation.drop(columns='bankrupt')
y_validation = validation['bankrupt']
print('Training rows:', len(train))
print('Validation rows:', len(validation))
print('Bankrupt companies in validation:', int(y_validation.sum()))

Training rows: 4861
Validation rows: 1042
Bankrupt companies in validation: 41


## 2. Fitting the models

The majority benchmark always predicts non-bankruptcy. 

Logistic regression (and trees) learn from the financial ratios. Its features are scaled using the training set, the same scaling is then applied to validation data. 

In [36]:
majority = DummyClassifier(strategy='most_frequent')
majority.fit(X_train, y_train)
logistic = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)
logistic.fit(X_train, y_train)
majority_predictions = majority.predict(X_validation)
logistic_predictions = logistic.predict(X_validation)

## 3. Produce results

Recall measures how many bankrupt companies were correctly identified. 
Precision measures how many companies predicted bankrupt actually were bankrupt. 
F1 combines precision and recall. 

ROC-AUC measures how well the model ranks bankrupt companies above non-bankrupt companies. Average precision summarizes the tradeoff between precision and recall across different cutoffs.

In [37]:
def scores(predictions, probabilities):
    return {
        'Accuracy': accuracy_score(y_validation, predictions),
        'Precision': precision_score(y_validation, predictions, zero_division=0),
        'Recall': recall_score(y_validation, predictions, zero_division=0),
        'F1': f1_score(y_validation, predictions, zero_division=0),
        'ROC-AUC': roc_auc_score(y_validation, probabilities),
        'Average precision': average_precision_score(y_validation, probabilities)
    }

results = pd.DataFrame({
    'Majority benchmark': scores(majority_predictions, majority.predict_proba(X_validation)[:, 1]),
    'Logistic regression': scores(logistic_predictions, logistic.predict_proba(X_validation)[:, 1])
}).T
results.round(3)

,Accuracy,Precision,Recall,F1,ROC-AUC,Average precision
Majority benchmark,0.961,0.000,0.000,0.000,0.500,0.039
Logistic regression,0.960,0.333,0.024,0.045,0.674,0.109


## 4.  Further analysis of logistic model 

Rows show the actual class and columns show the predicted class. The bottom right value is the number of bankrupt companies correctly identified.

In [38]:
pd.DataFrame(
    confusion_matrix(y_validation, logistic_predictions, labels=[0, 1]),
    index=['Actual non-bankrupt', 'Actual bankrupt'],
    columns=['Predicted non-bankrupt', 'Predicted bankrupt']
)

,Predicted non-bankrupt,Predicted bankrupt
Actual non-bankrupt,999,2
Actual bankrupt,40,1


## 5. Class-weighted logistic regression

This version gives more weight to the less common class (bankrupt) while training. We use the same training and validation sets as earlier, and compare both logistic models using recall, precision, and F1. We hope this will produce better results due to the class imbalance.


In [39]:
weighted_logistic = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
)
weighted_logistic.fit(X_train, y_train)
weighted_predictions = weighted_logistic.predict(X_validation)
comparison = results.copy()
comparison.loc['Weighted logistic regression'] = scores(weighted_predictions, weighted_logistic.predict_proba(X_validation)[:, 1])
comparison.round(3)


,Accuracy,Precision,Recall,F1,ROC-AUC,Average precision
Majority benchmark,0.961,0.000,0.000,0.000,0.500,0.039
Logistic regression,0.960,0.333,0.024,0.045,0.674,0.109
Weighted logistic regression,0.674,0.072,0.610,0.128,0.720,0.128


The table below shows how many validation companies the weighted model classified correctly and incorrectly.


In [40]:
pd.DataFrame(
    confusion_matrix(y_validation, weighted_predictions, labels=[0, 1]),
    index=['Actual non-bankrupt', 'Actual bankrupt'],
    columns=['Predicted non-bankrupt', 'Predicted bankrupt']
)


,Predicted non-bankrupt,Predicted bankrupt
Actual non-bankrupt,677,324
Actual bankrupt,16,25


## 6. Decision tree

This first tree uses the default settings and no class weighting. It does not need any type of scaling. We will compare it using the same validation set.


In [41]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)
tree_predictions = tree.predict(X_validation)

comparison.loc['Decision tree'] = scores(tree_predictions, tree.predict_proba(X_validation)[:, 1])
comparison.round(3)


,Accuracy,Precision,Recall,F1,ROC-AUC,Average precision
Majority benchmark,0.961,0.000,0.000,0.000,0.500,0.039
Logistic regression,0.960,0.333,0.024,0.045,0.674,0.109
Weighted logistic regression,0.674,0.072,0.610,0.128,0.720,0.128
Decision tree,0.942,0.327,0.439,0.375,0.701,0.166


The rows show the actual class and the columns show what the decision tree predicted.


In [42]:
pd.DataFrame(
    confusion_matrix(y_validation, tree_predictions, labels=[0, 1]),
    index=['Actual non-bankrupt', 'Actual bankrupt'],
    columns=['Predicted non-bankrupt', 'Predicted bankrupt']
)


,Predicted non-bankrupt,Predicted bankrupt
Actual non-bankrupt,964,37
Actual bankrupt,23,18


## 7. Trying different tree depths

 We will try three maximum depths and compare them with the original tree using the same training and validation data. 


In [43]:
depth_results = []
for depth in [3, 5, 8]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    predictions = model.predict(X_validation)
    tn, fp, fn, tp = confusion_matrix(y_validation, predictions, labels=[0, 1]).ravel()
    depth_results.append({
        'Model': f'Tree (max depth {depth})',
        **scores(predictions, model.predict_proba(X_validation)[:, 1]),
        'Bankrupt found': tp,
        'False positives': fp
    })
depth_comparison = pd.DataFrame(depth_results).set_index('Model')
depth_comparison.round(3)


,Accuracy,Precision,Recall,F1,ROC-AUC,Average precision,Bankrupt found,False positives
Model,,,,,,,,
Tree (max depth 3),0.966,0.750,0.220,0.340,0.708,0.224,9,3
Tree (max depth 5),0.961,0.500,0.244,0.328,0.774,0.304,10,10
Tree (max depth 8),0.965,0.593,0.390,0.471,0.759,0.287,16,11


## 8. Class-weighted decision trees

Class weighting gives the less common bankrupt class more importance during training. We try the same three depth limits as before and an unrestricted tree, then compare the results on the validation set. 


In [44]:
weighted_tree_results = []
for depth in [3, 5, 8, None]:
    model = DecisionTreeClassifier(max_depth=depth, class_weight='balanced', random_state=42)
    model.fit(X_train, y_train)
    predictions = model.predict(X_validation)
    tn, fp, fn, tp = confusion_matrix(y_validation, predictions, labels=[0, 1]).ravel()
    weighted_tree_results.append({
        'Model': f'Weighted tree (max depth {depth})',
        **scores(predictions, model.predict_proba(X_validation)[:, 1]),
        'Bankrupt found': tp,
        'False positives': fp
    })
weighted_tree_comparison = pd.DataFrame(weighted_tree_results).set_index('Model')
weighted_tree_comparison.round(3)


,Accuracy,Precision,Recall,F1,ROC-AUC,Average precision,Bankrupt found,False positives
Model,,,,,,,,
Weighted tree (max depth 3),0.716,0.098,0.756,0.173,0.775,0.241,31,286
Weighted tree (max depth 5),0.829,0.134,0.610,0.219,0.777,0.387,25,162
Weighted tree (max depth 8),0.916,0.230,0.488,0.312,0.712,0.323,20,67
Weighted tree (max depth None),0.958,0.457,0.390,0.421,0.686,0.202,16,19


## 9. Milestone 3 results table

These are results for the cleaned `1year` validation set. The first two models are the naive benchmark and logistic regression (our course-level baseline). We chose to show the depth-5 weighted tree because it balances catching bankrupt companies with limiting false positives.

In [45]:
baseline_table = comparison.loc[['Majority benchmark', 'Logistic regression']].copy()
baseline_table['False positives'] = [
    confusion_matrix(y_validation, majority_predictions, labels=[0, 1])[0, 1],
    confusion_matrix(y_validation, logistic_predictions, labels=[0, 1])[0, 1]
]
weighted_depth5 = weighted_tree_comparison.loc['Weighted tree (max depth 5)']
baseline_table.loc['Weighted tree (depth 5)'] = weighted_depth5[baseline_table.columns]
baseline_table['False positives'] = baseline_table['False positives'].astype(int)
baseline_table.round(3)


,Accuracy,Precision,Recall,F1,ROC-AUC,Average precision,False positives
Majority benchmark,0.961,0.000,0.000,0.000,0.500,0.039,0
Logistic regression,0.960,0.333,0.024,0.045,0.674,0.109,2
Weighted tree (depth 5),0.829,0.134,0.610,0.219,0.777,0.387,162


### What these results show at a glance

- Basic logistic regression identifies very few bankrupt companies in this validation set.
- The class-weighted depth-5 tree finds more bankrupt companies but also incorrectly labels many non-bankrupt companies as bankrupt.

These results are preliminary. The goal is to establish a baseline for future comparison.